Save as: module10-simulation-lab.ipynb

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncap/micro-course/blob/main/module-10/module10-simulation-lab.ipynb)

# Lab 10 — The Audit Battery
### Stability as a measured property of your pilot
*Microeconomic Theory for Experimentalists & Applied Microeconomists*

**This lab is a harness, not an experiment.** It runs any agent task
across **models × content-preserving paraphrases × temperatures** and
produces the stability report: per-cell means, paraphrase ranges (the
prompt-sensitivity index), cross-model gaps, and a claims ladder.

**Why:** "one model, one prompt" results are draws from a distribution
you never characterized. The battery characterizes it — and its output
is a **required appendix of your capstone paper**.

**The default task** is Module 1's dictator game, so the report has
something to show before you touch anything. **Your assignment:** swap
in YOUR capstone pilot's task, paraphrases, parser, and claims (the
CHANGE HERE cells), then run and read.

**The claims ladder (commit before running):** each instability knocks
claims off the top — effect size falls first, then heterogeneity, then
magnitude; sign/existence falls last. The human cross-lab band for the
default task (dictator means vary ±5 pp across comparable labs) is
your fragility reference.

**How to run:** cells top to bottom. In DRY_RUN, the mock simulates
two "models" with different biases and one loaded paraphrase, so the
report has structure to find.


## Setup and the DRY_RUN switch

The notebook starts in **DRY_RUN mode**: it executes end-to-end on synthetic
placeholder data, with no API key and no cost, so you can check your design
and see the analysis pipeline work.

> ⚠️ **DRY_RUN output is fake.** It is generated by `mock_response()`, not
> by a model. Never report it as a result.

When your design is ready, set `DRY_RUN = False` in the cell below and run
again in Colab — you will be asked for your Anthropic API key via a hidden
prompt (the key is never stored in the notebook file).


In [ ]:
%pip install anthropic pandas matplotlib --quiet

In [ ]:
import itertools
import json
import random
import re
import time

import pandas as pd

DRY_RUN = True   # <-- set to False in Colab to query the real model

MODEL = "claude-sonnet-5"   # swap models here to test robustness across models
TEMPERATURE = 1.0            # fixed and reported — part of the experimental design
N_PER_CELL = 30              # decisions per persona x treatment cell

if DRY_RUN:
    print("=" * 62)
    print("DRY RUN: synthetic placeholder data — NOT model behavior.")
    print("Set DRY_RUN = False to run the real experiment.")
    print("=" * 62)

    class _MockBlock:
        def __init__(self, text):
            self.text = text

    class _MockResponse:
        def __init__(self, text):
            self.content = [_MockBlock(text)]

    class _MockMessages:
        def create(self, **kwargs):
            # mock_response() is defined in the design cell below
            return _MockResponse(mock_response(kwargs["messages"][0]["content"]))

    class _MockClient:
        messages = _MockMessages()

    client = _MockClient()
else:
    from getpass import getpass

    import anthropic

    # Never hard-code API keys. getpass keeps the key out of the notebook file.
    API_KEY = getpass("Paste your Anthropic API key: ")
    client = anthropic.Anthropic(api_key=API_KEY)


## The battery configuration — CHANGE HERE (all four blocks live here)

Defaults: 2 models × 3 paraphrases × 2 temperatures × 15 runs = 180
calls. Swapping in your capstone task means replacing:
**(1)** `PARAPHRASES` — three renderings of YOUR task, same content,
different wording (have a classmate verify content-equivalence);
**(2)** `MODELS` and `TEMPERATURES` if your budget allows more;
**(3)** `parse_response` for your answer format; and
**(4)** `mock_response` so DRY_RUN still executes. The analysis cell's
claims table is also yours to fill.


In [ ]:
# Three content-preserving renderings of the task.  CHANGE HERE (1 of 4)
# Default: the Module 1 dictator game. Paraphrase 3 deliberately uses
# warmer, norm-adjacent wording — the kind of drift the battery exists
# to catch.
ENDOWMENT = 10
PARAPHRASES = {
    "p1_neutral": (
        "You are an adult participating in a paid decision study. You "
        f"have been given ${ENDOWMENT}. You may give any whole-dollar "
        f"amount between $0 and ${ENDOWMENT} to another anonymous "
        "participant. You keep the rest. Decide as you genuinely "
        'would.\n\nRespond with ONLY a JSON object: '
        '{"give": <integer 0-10>}.'
    ),
    "p2_reordered": (
        "You are an adult in a paid decision study. Another anonymous "
        "participant has been paired with you. You received "
        f"${ENDOWMENT}; any whole-dollar amount from $0 to "
        f"${ENDOWMENT} may be passed to them, and whatever you do not "
        "pass is yours. Decide as you genuinely would.\n\n"
        'Respond with ONLY a JSON object: {"give": <integer 0-10>}.'
    ),
    "p3_warm": (
        "You are an adult taking part in a paid study with a partner "
        f"you will never meet. You have ${ENDOWMENT} to share between "
        "the two of you however you wish, in whole dollars; your "
        "partner gets what you give, you keep the rest. Decide as you "
        'genuinely would.\n\nRespond with ONLY a JSON object: '
        '{"give": <integer 0-10>}.'
    ),
}

MODELS_TO_AUDIT = ["claude-sonnet-5", "claude-haiku-4-5-20251001"]  # CHANGE HERE (2 of 4)
TEMPERATURES = [0.7, 1.0]
N_PER_CELL_BATTERY = 15


def parse_response(text: str) -> "dict | None":  # CHANGE HERE (3 of 4)
    match = re.search(r'\{\s*"give"\s*:\s*(\d+)\s*\}', text)
    if match:
        give = int(match.group(1))
        if 0 <= give <= ENDOWMENT:
            return {"give": give}
    return None


def mock_response(prompt: str, model: str = "") -> str:  # CHANGE HERE (4 of 4)
    """DRY_RUN only: simulates two 'models' with different biases and a
    loaded third paraphrase, so the stability report has structure to
    find. NOT model behavior. Replace alongside your task."""
    center = 2.8 if "haiku" in model else 3.3
    if "to share between" in prompt:              # the warm paraphrase
        center += 1.2 if "haiku" not in model else 0.3
    give = int(round(min(max(random.gauss(center, 1.2), 0), ENDOWMENT)))
    return json.dumps({"give": give})


## The battery engine *(do not modify)*

Loops models × paraphrases × temperatures; in DRY_RUN the mock client
is re-wired per audited model so the report has cross-model structure.
Saves `lab10_results.csv` and `lab10_prompts_log.json`.


In [ ]:
def battery_call(prompt, model, temperature):
    if DRY_RUN:
        return mock_response(prompt, model), "(dry run)"
    try:
        response = client.messages.create(
            model=model, max_tokens=100, temperature=temperature,
            messages=[{"role": "user", "content": prompt}],
        )
        raw = response.content[0].text
        return raw, raw
    except Exception as err:
        time.sleep(5)
        return None, f"ERROR: {err}"


def run_battery() -> pd.DataFrame:
    rows = []
    cells = [(m, p, t) for m in MODELS_TO_AUDIT
             for p in PARAPHRASES for t in TEMPERATURES]
    print(f"{len(cells)} cells x {N_PER_CELL_BATTERY} runs = "
          f"{len(cells) * N_PER_CELL_BATTERY} calls")
    for model, para, temp in cells:
        prompt = PARAPHRASES[para]
        for i in range(N_PER_CELL_BATTERY):
            text, raw = battery_call(prompt, model, temp)
            decision = parse_response(text) if text else None
            rows.append({
                "model": model, "paraphrase": para, "temperature": temp,
                "rep": i,
                "give": None if decision is None else decision["give"],
                "raw": raw,
            })
        done = sum(1 for r in rows if r["give"] is not None
                   and r["model"] == model and r["paraphrase"] == para
                   and r["temperature"] == temp)
        print(f"  {model} | {para} | T={temp}: "
              f"{done}/{N_PER_CELL_BATTERY} parsed")
    df = pd.DataFrame(rows)
    df.to_csv("lab10_results.csv", index=False)
    with open("lab10_prompts_log.json", "w") as f:
        json.dump(PARAPHRASES, f, indent=2)
    return df


df = run_battery()


## The stability report *(fill the claims table; don't modify the rest)*

Three statistics, then the verdicts: **paraphrase range** per model
(the prompt-sensitivity index, against the human cross-lab band),
**cross-model gap** per paraphrase, and **temperature spread**. Then
the claims ladder — replace the default claims with YOUR pilot's.


In [ ]:
HUMAN_CROSS_LAB_BAND = 5.0   # +/- pp; dictator-game default — cite yours

valid = df.dropna(subset=["give"]).copy()
valid["pct"] = valid["give"] * 10.0   # percent of endowment
print("Parse-failure rate by cell:")
fail = df.assign(f=df["give"].isna()).pivot_table(
    index="model", columns="paraphrase", values="f", aggfunc="mean").round(3)
print(fail)

print("\n=== Cell means (percent given) ===")
table = valid.pivot_table(index=["model", "temperature"],
                          columns="paraphrase", values="pct",
                          aggfunc="mean").round(1)
print(table)

print("\n=== Paraphrase range per model (prompt-sensitivity index) ===")
by_para = valid.pivot_table(index="model", columns="paraphrase",
                            values="pct", aggfunc="mean")
ranges = (by_para.max(axis=1) - by_para.min(axis=1)).round(1)
print(ranges)
print(f"Reference: human cross-lab band ~ +/-{HUMAN_CROSS_LAB_BAND} pp "
      f"(width {2 * HUMAN_CROSS_LAB_BAND:.0f}). Ranges beyond it are an "
      "instrument property — they cap your claims.")

print("\n=== Cross-model gap per paraphrase ===")
by_model = valid.pivot_table(index="paraphrase", columns="model",
                             values="pct", aggfunc="mean")
gaps = (by_model.max(axis=1) - by_model.min(axis=1)).round(1)
print(gaps)

print("\n=== Temperature spread (max - min across T, per model) ===")
by_temp = valid.pivot_table(index="model", columns="temperature",
                            values="pct", aggfunc="mean")
print((by_temp.max(axis=1) - by_temp.min(axis=1)).round(1))

# ----------------- THE CLAIMS LADDER — fill with YOUR pilot's claims ---------
CLAIMS = [
    "giving is positive and substantial",            # sign/existence
    "mean giving is roughly 30 percent",             # magnitude
    "giving differs across personas as predicted",   # heterogeneity
    "the effect size is X",                          # effect size
]
print("\n=== Claims ladder (verdicts are YOURS to argue in Section 5) ===")
for c in CLAIMS:
    print(f"  [ SURVIVES / DOES NOT SURVIVE ]  {c}")
print("\nRule of thumb: effect size falls first, then heterogeneity, "
      "then magnitude; sign falls last. Over-claiming costs double.")


## This lab IS the robustness section

The battery is the course's per-lab robustness conventions (paraphrase,
second model), promoted to the main event and made quantitative. Two
extensions if budget allows: a third model from a different provider
(the provider-agnostic wrapper exists for this), and a
**contamination probe** — one preregistered out-of-corpus variation of
your task (PS10's Bayes logic says this single run carries more
evidential weight than every famous-result replication combined).

**Limits of silicon subjects — the final box:** the battery measures
prompt fragility and cross-model instability, which bounds your
claims; it does NOT certify fidelity (a stable answer can be a stably
memorized answer — stability is necessary, never sufficient). What
certifies uses is the can/cannot list: instruments, pipelines, power
machinery, framing detection, hypothesis generation, and deployed-AI
audits — with effect sizes, welfare claims, and calibration rung 4
permanently off the table. Your capstone's honest methods paragraph
states which list every use came from.
